In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib numpy qiskit-addon-obp qiskit-addon-utils rustworkx
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Rétropropagation d'opérateurs (OBP) pour l'estimation des valeurs attendues

*Estimation d'utilisation : 16 minutes sur un processeur Eagle r3 (REMARQUE : Ceci est uniquement une estimation. Votre temps d'exécution peut varier.)*

In [1]:
import numpy as np
import matplotlib.pyplot as plt

from qiskit.primitives import StatevectorEstimator
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit.quantum_info import SparsePauliOp
from qiskit.transpiler import CouplingMap
from qiskit.synthesis import LieTrotter

from qiskit_addon_utils.problem_generators import generate_xyz_hamiltonian
from qiskit_addon_utils.problem_generators import (
    generate_time_evolution_circuit,
)
from qiskit_addon_utils.slicing import slice_by_depth, combine_slices
from qiskit_addon_obp.utils.simplify import OperatorBudget
from qiskit_addon_obp import backpropagate
from qiskit_addon_obp.utils.truncating import setup_budget

from rustworkx.visualization import graphviz_draw

from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2, EstimatorOptions

## Contexte
La rétropropagation d'opérateurs est une technique qui consiste à absorber des opérations depuis la fin d'un circuit quantique dans l'observable mesuré, réduisant généralement la profondeur du circuit au prix de termes supplémentaires dans l'observable. L'objectif est de rétropropager autant du circuit que possible sans permettre à l'observable de devenir trop volumineux. Une implémentation basée sur Qiskit est disponible dans l'addon OBP de Qiskit ; vous trouverez plus de détails dans la [documentation](/guides/qiskit-addons-obp) correspondante avec un [exemple simple](/guides/qiskit-addons-obp-get-started) pour démarrer.

Considérons un exemple de circuit pour lequel un observable $O = \sum_P c_P P$ doit être mesuré, où $P$ sont des opérateurs de Pauli et $c_P$ sont des coefficients. Notons le circuit comme un unitaire unique $U$ qui peut être logiquement partitionné en $U = U_C U_Q$ comme illustré dans la figure ci-dessous.

![Circuit diagram showing Uq followed by Uc](../docs/images/tutorials/improving-estimation-of-expectation-values-with-operator-backpropagation/logical-partitioning.avif)

La rétropropagation d'opérateurs absorbe l'unitaire $U_C$ dans l'observable en le faisant évoluer comme $O' = U_C^{\dagger}OU_C = \sum_P c_P U_C^{\dagger}PU_C$. Autrement dit, une partie du calcul est effectuée classiquement via l'évolution de l'observable de $O$ à $O'$. Le problème original peut maintenant être reformulé comme la mesure de l'observable $O'$ pour le nouveau circuit de profondeur réduite dont l'unitaire est $U_Q$.

L'unitaire $U_C$ est représenté sous forme d'un certain nombre de tranches $U_C = U_S U_{S-1}...U_2U_1$. Il existe plusieurs façons de définir une tranche. Par exemple, dans le circuit ci-dessus, chaque couche de portes $R_{zz}$ et chaque couche de portes $R_x$ peut être considérée comme une tranche individuelle. La rétropropagation implique le calcul classique de $O' = \Pi_{s=1}^S \sum_P c_P U_s^{\dagger} P U_s$. Chaque tranche $U_s$ peut être représentée comme $U_s = exp(\frac{-i\theta_s P_s}{2})$, où $P_s$ est un opérateur de Pauli à $n$ qubits et $\theta_s$ est un scalaire. Il est facile de vérifier que

$$
U_s^{\dagger} P U_s = P \qquad \text{if} ~[P,P_s] = 0,
$$
$$
U_s^{\dagger} P U_s = \qquad cos(\theta_s)P + i sin(\theta_s)P_sP \qquad \text{if} ~{P,P_s} = 0
$$

Dans l'exemple ci-dessus, si ${P,P_s} = 0$, alors nous devons exécuter deux circuits quantiques, au lieu d'un seul, pour calculer la valeur attendue. Par conséquent, la rétropropagation peut augmenter le nombre de termes dans l'observable, entraînant un nombre plus élevé d'exécutions de circuits. Une façon de permettre une rétropropagation plus profonde dans le circuit, tout en empêchant l'opérateur de devenir trop volumineux, est de tronquer les termes avec de petits coefficients, plutôt que de les ajouter à l'opérateur. Par exemple, dans le cas ci-dessus, on peut choisir de tronquer le terme impliquant $P_sP$ à condition que $\theta_s$ soit suffisamment petit. La troncature de termes peut entraîner moins de circuits quantiques à exécuter, mais cela introduit une certaine erreur dans le calcul final de la valeur attendue, proportionnelle à la magnitude des coefficients des termes tronqués.

Ce tutoriel implémente un [pattern Qiskit](/guides/intro-to-patterns) pour simuler la dynamique quantique d'une chaîne de spins de Heisenberg en utilisant <a href="https://github.com/Qiskit/qiskit-addon-obp">qiskit-addon-obp</a>.
## Prérequis
Avant de commencer ce tutoriel, assurez-vous d'avoir installé les éléments suivants :

- Qiskit SDK v1.2 ou ultérieur (`pip install qiskit`)
- Qiskit Runtime v0.28 ou ultérieur (`pip install qiskit-ibm-runtime`)
- Addon OBP de Qiskit (`pip install qiskit-addon-obp`)
- Utilitaires des addons Qiskit (`pip install qiskit-addon-utils`)
## Configuration

In [2]:
num_qubits = 10
layout = [(i - 1, i) for i in range(1, num_qubits)]

# Instantiate a CouplingMap object
coupling_map = CouplingMap(layout)
graphviz_draw(coupling_map.graph, method="circo")

<Image src="../docs/images/tutorials/operator-back-propagation/extracted-outputs/a3debf65-06df-4277-933e-14b6f6170756-0.avif" alt="Output of the previous code cell" />

## Partie I : Chaîne de spins de Heisenberg à petite échelle
### Étape 1 : Mapper les entrées classiques vers un problème quantique
#### Mapper l'évolution temporelle d'un modèle quantique de Heisenberg vers une expérience quantique.
Le package [qiskit_addon_utils](https://github.com/Qiskit/qiskit-addon-utils) fournit des fonctionnalités réutilisables à diverses fins.

Son module [qiskit_addon_utils.problem_generators](https://docs.quantum.ibm.com/api/qiskit-addon-utils/problem-generators) fournit des fonctions pour générer des hamiltoniens de type Heisenberg sur un graphe de connectivité donné.
Ce graphe peut être soit un [rustworkx.PyGraph](https://www.rustworkx.org/apiref/rustworkx.PyGraph.html), soit un [CouplingMap](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.CouplingMap), ce qui facilite son utilisation dans les flux de travail centrés sur Qiskit.

Dans ce qui suit, nous générons un `CouplingMap` en chaîne linéaire de 10 qubits.

In [3]:
# Get a qubit operator describing the Heisenberg XYZ model
hamiltonian = generate_xyz_hamiltonian(
    coupling_map,
    coupling_constants=(np.pi / 8, np.pi / 4, np.pi / 2),
    ext_magnetic_field=(np.pi / 3, np.pi / 6, np.pi / 9),
)
print(hamiltonian)

SparsePauliOp(['IIIIIIIXXI', 'IIIIIIIYYI', 'IIIIIIIZZI', 'IIIIIXXIII', 'IIIIIYYIII', 'IIIIIZZIII', 'IIIXXIIIII', 'IIIYYIIIII', 'IIIZZIIIII', 'IXXIIIIIII', 'IYYIIIIIII', 'IZZIIIIIII', 'IIIIIIIIXX', 'IIIIIIIIYY', 'IIIIIIIIZZ', 'IIIIIIXXII', 'IIIIIIYYII', 'IIIIIIZZII', 'IIIIXXIIII', 'IIIIYYIIII', 'IIIIZZIIII', 'IIXXIIIIII', 'IIYYIIIIII', 'IIZZIIIIII', 'XXIIIIIIII', 'YYIIIIIIII', 'ZZIIIIIIII', 'IIIIIIIIIX', 'IIIIIIIIIY', 'IIIIIIIIIZ', 'IIIIIIIIXI', 'IIIIIIIIYI', 'IIIIIIIIZI', 'IIIIIIIXII', 'IIIIIIIYII', 'IIIIIIIZII', 'IIIIIIXIII', 'IIIIIIYIII', 'IIIIIIZIII', 'IIIIIXIIII', 'IIIIIYIIII', 'IIIIIZIIII', 'IIIIXIIIII', 'IIIIYIIIII', 'IIIIZIIIII', 'IIIXIIIIII', 'IIIYIIIIII', 'IIIZIIIIII', 'IIXIIIIIII', 'IIYIIIIIII', 'IIZIIIIIII', 'IXIIIIIIII', 'IYIIIIIIII', 'IZIIIIIIII', 'XIIIIIIIII', 'YIIIIIIIII', 'ZIIIIIIIII'],
              coeffs=[0.39269908+0.j, 0.78539816+0.j, 1.57079633+0.j, 0.39269908+0.j,
 0.78539816+0.j, 1.57079633+0.j, 0.39269908+0.j, 0.78539816+0.j,
 1.57079633+0.j, 0.39269908+0.j, 0.

![Output of the previous code cell](../docs/images/tutorials/operator-back-propagation/extracted-outputs/de8ce35e-a2c5-474f-adb9-88c9afb2bd76-0.avif)

Ensuite, nous générons un opérateur de Pauli modélisant un hamiltonien XYZ de Heisenberg.

$$
{\hat{\mathcal{H}}_{XYZ} = \sum_{(j,k)\in E} (J_{x} \sigma_j^{x} \sigma_{k}^{x} + J_{y} \sigma_j^{y} \sigma_{k}^{y} + J_{z} \sigma_j^{z} \sigma_{k}^{z}) + \sum_{j\in V} (h_{x} \sigma_j^{x} + h_{y} \sigma_j^{y} + h_{z} \sigma_j^{z})}
$$

Où $G(V,E)$ est le graphe du coupling map fourni.

In [4]:
circuit = generate_time_evolution_circuit(
    hamiltonian,
    time=0.2,
    synthesis=LieTrotter(reps=2),
)
circuit.draw("mpl", style="iqp", fold=-1)

<Image src="../docs/images/tutorials/operator-back-propagation/extracted-outputs/5208e0a8-0.avif" alt="Output of the previous code cell" />

### Step 2: Optimize problem for quantum hardware execution

#### Create circuit slices to backpropagate

The `backpropagate` function backpropagates entire circuit slices at a time. Therefore, the choice of slicing can have an impact on how well backpropagation performs for a given problem. Here, we will group gates of the same type into slices using the [`slice_by_depth`](/docs/api/qiskit-addon-utils/slicing#slice_by_depth) function.

For a more detailed discussion on circuit slicing, check out this [how-to guide](https://qiskit.github.io/qiskit-addon-utils/how_tos/create_circuit_slices.html) of the [`qiskit-addon-utils`](https://github.com/Qiskit/qiskit-addon-utils) package.

In [5]:
slices = slice_by_depth(circuit, max_slice_depth=1)
print(f"Separated the circuit into {len(slices)} slices.")

Separated the circuit into 18 slices.


À partir de l'opérateur de qubit, nous pouvons générer un circuit quantique qui modélise son évolution temporelle.
Encore une fois, le module [qiskit_addon_utils.problem_generators](https://docs.quantum.ibm.com/api/qiskit-addon-utils/problem-generators) vient à la rescousse avec une fonction pratique pour cela :

In [6]:
op_budget = OperatorBudget(max_qwc_groups=8)

![Output of the previous code cell](../docs/images/tutorials/operator-back-propagation/extracted-outputs/1d68f197-ffa4-49de-9fe8-243b1facbd00-0.avif)

### Étape 2 : Optimiser le problème pour l'exécution sur matériel quantique
#### Créer des tranches de circuit pour la rétropropagation
Rappelez-vous, la fonction ``backpropagate`` rétropropagera des tranches entières de circuit à la fois, donc le choix de la façon de découper peut avoir un impact sur les performances de la rétropropagation pour un problème donné. Ici, nous regrouperons les portes du même type en tranches en utilisant la fonction [slice_by_gate_types](https://docs.quantum.ibm.com/api/qiskit-addon-utils/slicing#slice_by_gate_types).

Pour une discussion plus détaillée sur le découpage de circuits, consultez ce [guide pratique](https://qiskit.github.io/qiskit-addon-utils/how_tos/create_circuit_slices.html) du package [qiskit-addon-utils](https://github.com/Qiskit/qiskit-addon-utils).

In [7]:
observable = SparsePauliOp.from_sparse_list(
    [("Z", [i], 1 / num_qubits) for i in range(num_qubits)],
    num_qubits=num_qubits,
)
observable

SparsePauliOp(['IIIIIIIIIZ', 'IIIIIIIIZI', 'IIIIIIIZII', 'IIIIIIZIII', 'IIIIIZIIII', 'IIIIZIIIII', 'IIIZIIIIII', 'IIZIIIIIII', 'IZIIIIIIII', 'ZIIIIIIIII'],
              coeffs=[0.1+0.j, 0.1+0.j, 0.1+0.j, 0.1+0.j, 0.1+0.j, 0.1+0.j, 0.1+0.j, 0.1+0.j,
 0.1+0.j, 0.1+0.j])

Below you will see that we backpropagated six slices, and the terms were combined into six and not eight groups. This implies that backpropagating one more slice would cause the number of Pauli groups to exceed eight. We can verify that this is the case by inspecting the returned metadata. Also note that in this portion the circuit transformation is exact.  That is, no terms of the new observable $O’$ were truncated. The backpropagated circuit and the backpropagated operator give the exact outcome as the original circuit and operator.

In [8]:
# Backpropagate slices onto the observable
bp_obs, remaining_slices, metadata = backpropagate(
    observable, slices, operator_budget=op_budget
)
# Recombine the slices remaining after backpropagation
bp_circuit = combine_slices(remaining_slices)

print(f"Backpropagated {metadata.num_backpropagated_slices} slices.")
print(
    f"New observable has {len(bp_obs.paulis)} terms, which can be combined into {len(bp_obs.group_commuting(qubit_wise=True))} groups."
)
print(
    f"Note that backpropagating one more slice would result in {metadata.backpropagation_history[-1].num_paulis[0]} terms "
    f"across {metadata.backpropagation_history[-1].num_qwc_groups} groups."
)
print("The remaining circuit after backpropagation looks as follows:")
bp_circuit.draw("mpl", fold=-1, scale=0.6)

Backpropagated 6 slices.
New observable has 60 terms, which can be combined into 6 groups.
Note that backpropagating one more slice would result in 114 terms across 12 groups.
The remaining circuit after backpropagation looks as follows:


<Image src="../docs/images/tutorials/operator-back-propagation/extracted-outputs/ee8fd385-1.avif" alt="Output of the previous code cell" />

#### Contraindre la taille maximale de l'opérateur pendant la rétropropagation
Pendant la rétropropagation, le nombre de termes dans l'opérateur approchera généralement $4^N$ rapidement, où $N$ est le nombre de qubits. Lorsque deux termes de l'opérateur ne commutent pas qubit par qubit, nous avons besoin de circuits séparés pour obtenir les valeurs attendues correspondantes. Par exemple, si nous avons un observable à 2 qubits $O = 0.1 XX + 0.3 IZ - 0.5 IX$, alors puisque $[XX,IX] = 0$, une mesure dans une seule base suffit pour calculer les valeurs attendues de ces deux termes. Cependant, $IZ$ anti-commute avec les deux autres termes. Nous avons donc besoin d'une mesure de base séparée pour calculer la valeur attendue de $IZ$. En d'autres termes, nous avons besoin de deux circuits, au lieu d'un seul, pour calculer $\langle O \rangle$. À mesure que le nombre de termes dans l'opérateur augmente, il est possible que le nombre requis d'exécutions de circuits augmente également.

La taille de l'opérateur peut être bornée en spécifiant le paramètre ``operator_budget`` de la fonction ``backpropagate``, qui accepte une instance de [OperatorBudget](https://docs.quantum.ibm.com/api/qiskit-addon-obp/utils-simplify#operatorbudget).

Pour contrôler la quantité de ressources supplémentaires (temps) allouées, nous restreignons le nombre maximal de groupes de Pauli commutant qubit par qubit que l'observable rétropropagé est autorisé à avoir. Ici, nous spécifions que la rétropropagation doit s'arrêter lorsque le nombre de groupes de Pauli commutant qubit par qubit dans l'opérateur dépasse 8.

In [ ]:
service = QiskitRuntimeService()
backend = service.least_busy(
    operational=True, simulator=False, min_num_qubits=133
)
print(backend)

<IBMBackend('ibm_kingston')>


In [10]:
pm_basis = generate_preset_pass_manager(
    optimization_level=3, basis_gates=backend.configuration().basis_gates
)
isa_circuit = pm_basis.run(circuit)
isa_bp_circuit = pm_basis.run(bp_circuit)

### Step 3: Execute using Qiskit primitives

First, we create two [Primitive Unified Blocs](/docs/api/qiskit/primitives) (PUBs) corresponding to the original circuit, and the backpropagated circuit. Then we execute the pubs on an ideal Estimator to obtain the expectation values.

In [11]:
pubs = [(isa_circuit, observable), (isa_bp_circuit, bp_obs)]

In [12]:
rng = np.random.default_rng()
estimator = StatevectorEstimator(seed=rng)
job = estimator.run(pubs)

### Step 4: Post-process and return result in desired classical format

Now we obtain the expectation values of the original and backpropagated circuits.

In [13]:
primitive_result = job.result()
circuit_expval = primitive_result[0].data.evs.item()
bp_circuit_expval = primitive_result[1].data.evs.item()

In [14]:
methods = [
    "No backpropagation",
    "Backpropagation",
]
values = [circuit_expval, bp_circuit_expval]

ax = plt.gca()
plt.bar(methods, values, color="#a56eff", width=0.4, edgecolor="#8a3ffc")
ax.set_ylim([0.6, 0.92])
ax.set_ylabel(r"$M_Z$", fontsize=12)

Text(0, 0.5, '$M_Z$')

<Image src="../docs/images/tutorials/operator-back-propagation/extracted-outputs/fb5f955a-1.avif" alt="Output of the previous code cell" />

Notez qu'en allouant `5e-3` d'erreur par tranche pour la troncature, nous sommes en mesure de supprimer 1 tranche supplémentaire du circuit, tout en restant dans le budget original de huit groupes de Pauli commutant dans l'observable. Par défaut, `backpropagate` utilise la norme L1 des coefficients tronqués pour borner l'erreur totale due à la troncature. Pour d'autres options, consultez le [guide pratique sur la spécification de la p_norm](https://qiskit.github.io/qiskit-addon-obp/how_tos/bound_error_using_p_norm.html).

Dans cet exemple particulier où nous avons rétropropagé sept tranches, l'erreur totale de troncature ne devrait pas dépasser ``(5e-3 error/slice) * (7 slices) = 3.5e-2``.
Pour une discussion plus approfondie sur la répartition d'un budget d'erreur entre vos tranches, consultez [ce guide pratique](https://qiskit.github.io/qiskit-addon-obp/how_tos/truncate_operator_terms.html).

In [15]:
num_qubits = 50
layout = [(i - 1, i) for i in range(1, num_qubits)]

# Instantiate a CouplingMap object
coupling_map = CouplingMap(layout)

hamiltonian = generate_xyz_hamiltonian(
    coupling_map,
    coupling_constants=(np.pi / 8, np.pi / 4, np.pi / 2),
    ext_magnetic_field=(np.pi / 3, np.pi / 6, np.pi / 9),
)

# Generate a time evolution circuit for the Hamiltonian
circuit = generate_time_evolution_circuit(
    hamiltonian,
    time=0.2,
    synthesis=LieTrotter(reps=4),
)

# Define the observable to measure
observable = SparsePauliOp.from_sparse_list(
    [("Z", [i], 1 / num_qubits) for i in range(num_qubits)],
    num_qubits,
)

slices = slice_by_depth(circuit, max_slice_depth=1)

# Define the maximum number of qwc groups allowed in the backpropagated observable, and the truncation error budget
op_budget = OperatorBudget(max_qwc_groups=15)
truncation_error_budget = setup_budget(
    max_error_total=0.03, max_error_per_slice=0.005
)

# First backpropagation without truncation
bp_obs, remaining_slices, metadata = backpropagate(
    observable, slices, operator_budget=op_budget
)
bp_circuit = combine_slices(remaining_slices)

# Now backpropagate with truncation, using the same operator budget and the defined truncation error budget
bp_obs_trunc, remaining_slices_trunc, metadata = backpropagate(
    observable,
    slices,
    operator_budget=op_budget,
    truncation_error_budget=truncation_error_budget,
)
bp_circuit_trunc = combine_slices(
    remaining_slices_trunc, include_barriers=False
)

# Now we transpile the original circuit and the two backpropagated circuits, and apply the layout to the corresponding observables
pm = generate_preset_pass_manager(optimization_level=3, backend=backend)

isa_circuit = pm.run(circuit)
isa_bp_circuit = pm.run(bp_circuit)
isa_bp_circuit_trunc = pm.run(bp_circuit_trunc)

isa_observable = observable.apply_layout(isa_circuit.layout)
isa_bp_observable = bp_obs.apply_layout(isa_bp_circuit.layout)
isa_bp_observable_trunc = bp_obs_trunc.apply_layout(
    isa_bp_circuit_trunc.layout
)

# Compare the 2-qubit depth of each transpiled circuit to see how much depth backpropagation saved
print(
    f"2-qubit depth without backpropagation: {isa_circuit.depth(lambda x: x.operation.num_qubits == 2)}"
)
print(
    f"2-qubit depth with backpropagation: {isa_bp_circuit.depth(lambda x: x.operation.num_qubits == 2)}"
)
print(
    f"2-qubit depth with backpropagation and truncation: {isa_bp_circuit_trunc.depth(lambda x: x.operation.num_qubits == 2)}"
)

pubs = [
    (isa_circuit, isa_observable),
    (isa_bp_circuit, isa_bp_observable),
    (isa_bp_circuit_trunc, isa_bp_observable_trunc),
]

# Now we instantiate the Estimator primitive for the hardware with ZNE and measurement error mitigation
# and compute the three circuits and observables
options = EstimatorOptions()
options.default_precision = 0.01
options.resilience_level = 2
options.resilience.zne.noise_factors = [1, 1.2, 1.4]
options.resilience.zne.extrapolator = ["linear"]
estimator = EstimatorV2(mode=backend, options=options)

estimator.options.environment.job_tags = ["TUT_OBP"]
job = estimator.run(pubs)

# Retrieve the results and the standard deviations
result_no_bp = job.result()[0].data.evs.item()
result_bp = job.result()[1].data.evs.item()
result_bp_trunc = job.result()[2].data.evs.item()

std_no_bp = job.result()[0].data.stds.item()
std_bp = job.result()[1].data.stds.item()
std_bp_trunc = job.result()[2].data.stds.item()

2-qubit depth without backpropagation: 24
2-qubit depth with backpropagation: 20
2-qubit depth with backpropagation and truncation: 18


In [16]:
print(f"Expectation value without backpropagation: {result_no_bp}")
print(f"Backpropagated expectation value: {result_bp}")
print(f"Backpropagated expectation value with truncation: {result_bp_trunc}")

Expectation value without backpropagation: 0.9543907942381811
Backpropagated expectation value: 0.9445337385406468
Backpropagated expectation value with truncation: 0.934050286970965


In [17]:
# Plot the results
methods = [
    "No backpropagation",
    "Backpropagation",
    "Backpropagation w/ truncation",
]
values = [result_no_bp, result_bp, result_bp_trunc]
error_bars = [std_no_bp, std_bp, std_bp_trunc]

ax = plt.gca()
plt.bar(methods, values, color="#a56eff", width=0.4, edgecolor="#8a3ffc")
plt.errorbar(methods, values, yerr=error_bars, fmt="o", color="r", capsize=5)
plt.axhline(0.89)
ax.set_ylim([0.8, 0.98])
plt.text(0.25, 0.895, "Exact result")
ax.set_ylabel(r"$M_Z$", fontsize=12)

Text(0, 0.5, '$M_Z$')

<Image src="../docs/images/tutorials/operator-back-propagation/extracted-outputs/37834c72-1.avif" alt="Output of the previous code cell" />

## Next steps
If you found this work interesting, you might be interested in the following material:
<Admonition type="tip" title="Recommendations">
- [Approximate quantum compilation for time evolution circuits](/docs/tutorials/approximate-quantum-compilation-for-time-evolution)
- [Multi-product formulas to reduce Trotter error](/docs/tutorials/multi-product-formula)
- [`pauli-prop`](https://github.com/Qiskit/pauli-prop), a Rust-accelerated package for Pauli propagation, with [tutorials](https://github.com/Qiskit/pauli-prop/tree/main/docs/tutorials) covering OBP, classical expectation-value estimation, and noisy simulation
</Admonition>